In [ ]:
import sqlite3, pandas as pd, random, time
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
DB_PATH = "/content/careerbot.db"
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Create tables
cursor.executescript("""
CREATE TABLE IF NOT EXISTS users (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  username TEXT UNIQUE,
  name TEXT,
  department TEXT,
  interests TEXT
);

CREATE TABLE IF NOT EXISTS skills (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  user_id INTEGER,
  skill TEXT,
  FOREIGN KEY(user_id) REFERENCES users(id)
);

CREATE TABLE IF NOT EXISTS projects (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  user_id INTEGER,
  project TEXT,
  FOREIGN KEY(user_id) REFERENCES users(id)
);

CREATE TABLE IF NOT EXISTS certifications (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  user_id INTEGER,
  certification TEXT,
  FOREIGN KEY(user_id) REFERENCES users(id)
);
""")
conn.commit()
print("✅ Database initialized at:", DB_PATH)

✅ Database initialized at: /content/careerbot.db


In [ ]:
def get_user(username):
    user = cursor.execute("SELECT * FROM users WHERE username=?", (username,)).fetchone()
    return user

def create_user(username, name, department, interests):
    cursor.execute("INSERT INTO users (username, name, department, interests) VALUES (?, ?, ?, ?)",
                   (username, name, department, interests))
    conn.commit()
    return cursor.lastrowid

def add_skill(user_id, skill):
    cursor.execute("INSERT INTO skills (user_id, skill) VALUES (?, ?)", (user_id, skill))
    conn.commit()

def add_project(user_id, project):
    cursor.execute("INSERT INTO projects (user_id, project) VALUES (?, ?)", (user_id, project))
    conn.commit()

def add_cert(user_id, cert):
    cursor.execute("INSERT INTO certifications (user_id, certification) VALUES (?, ?)", (user_id, cert))
    conn.commit()

def get_profile(user_id):
    profile = {}
    u = cursor.execute("SELECT name, department, interests FROM users WHERE id=?", (user_id,)).fetchone()
    profile["name"], profile["department"], profile["interests"] = u
    profile["skills"] = [x[0] for x in cursor.execute("SELECT skill FROM skills WHERE user_id=?", (user_id,))]
    profile["projects"] = [x[0] for x in cursor.execute("SELECT project FROM projects WHERE user_id=?", (user_id,))]
    profile["certifications"] = [x[0] for x in cursor.execute("SELECT certification FROM certifications WHERE user_id=?", (user_id,))]
    return profile

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model loaded


In [ ]:
def chatbot_reply(user_id, msg):
    prof = get_profile(user_id)
    name, dept = prof["name"], prof["department"]

    # Handle user adding new info
    if "learned" in msg.lower():
        skill = msg.split("learned")[-1].strip().title()
        add_skill(user_id, skill)
        return f"Nice, {name}! Added **{skill}** to your skill set 💪"

    if "project" in msg.lower() and "done" in msg.lower():
        project = msg.split("done")[-1].strip().title()
        add_project(user_id, project)
        return f"Cool! I’ve noted your new project: **{project}** 🔧"

    if "certification" in msg.lower():
        cert = msg.split("certification")[-1].strip().title()
        add_cert(user_id, cert)
        return f"Awesome, {name}! Added your certification **{cert}** 🎓"

    # Personalized small talk
    if "hello" in msg.lower() or "hi" in msg.lower():
        return f"Hey {name}! Great to see you again 👋. How’s everything going in {dept}?"

    # Default recommendation logic
    skills = ", ".join(prof["skills"])
    interests = prof["interests"]
    return (f"Alright {name}, since you’re from {dept} and skilled in {skills}, "
            f"you might enjoy exploring careers related to {interests} 🚀")

In [ ]:
# Login or Register
username = input("Enter your username: ").strip().lower()
user = get_user(username)

if not user:
    print("\n👋 Welcome new user! Let’s create your profile.")
    name = input("Enter your name: ").title()
    department = input("Enter your department: ").upper()
    skills = input("Enter your skills (comma separated): ")
    interests = input("Enter your interests: ")
    projects = input("Enter any projects done (comma separated): ")
    certifications = input("Enter any certifications (comma separated): ")

    user_id = create_user(username, name, department, interests)
    for s in skills.split(","): add_skill(user_id, s.strip().title())
    for p in projects.split(","): add_project(user_id, p.strip().title())
    for c in certifications.split(","): add_cert(user_id, c.strip().title())
    print(f"\n✅ Profile created for {name} ({department})!\n")
else:
    user_id = user[0]
    prof = get_profile(user_id)
    print(f"\nWelcome back, {prof['name']} 👋! I remember you’re from {prof['department']} "
          f"and already skilled in {', '.join(prof['skills'])}.\n")

# Chat loop
while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit", "bye"]:
        print("Bot: Bye! Have a great day 🚀")
        break
    reply = chatbot_reply(user_id, user_input)
    print("Bot:", reply, "\n")

KeyboardInterrupt: Interrupted by user

In [ ]:
cursor.execute("""
CREATE TABLE careers (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  career_name TEXT,
  category TEXT,
  required_skills TEXT,
  recommended_courses TEXT,
  projects TEXT,
  roadmap TEXT
);
""")
conn.commit()


In [ ]:
import json, sqlite3

conn = sqlite3.connect("careerbot.db")
cursor = conn.cursor()

with open("careers_dataset_50plus.json", "r") as f:
    data = json.load(f)

for row in data:
    cursor.execute("""
        INSERT INTO careers (career_name, category, required_skills, recommended_courses, projects, roadmap)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        row.get("career_name"),
        row.get("category"),
        json.dumps(row.get("required_skills")),
        json.dumps(row.get("recommended_courses")),
        json.dumps(row.get("projects")),
        row.get("roadmap", "")
    ))
conn.commit()

In [ ]:
import pandas as pd, sqlite3

conn = sqlite3.connect("/content/careerbot.db")
df = pd.read_sql("SELECT * FROM careers LIMIT 5", conn)
df.head()

,id,career_name,category,required_skills,recommended_courses,projects,roadmap
0,1,Data Scientist,Artificial Intelligence,"[""Python"", ""Machine Learning"", ""Statistics"", ""...","[""Machine Learning by Andrew Ng (Coursera)"", ""...","[""Customer Churn Prediction"", ""Sales Forecasti...",Start with Python and statistics → Learn data ...
1,2,Software Engineer,Computer Science,"[""Python"", ""Java"", ""C++"", ""Data Structures"", ""...","[""CS50: Introduction to Computer Science (edX)...","[""Portfolio Website"", ""Chat Application"", ""Lib...",Learn programming languages and DSA → Build sm...
2,3,Mechanical Design Engineer,Mechanical Engineering,"[""SolidWorks"", ""AutoCAD"", ""Finite Element Anal...","[""NPTEL: Design of Machine Elements"", ""CAD and...","[""3D Printer Design"", ""Gearbox Mechanism"", ""Ro...",Learn CAD tools → Master 3D modeling → Work wi...
3,4,Electrical Engineer,Electrical Engineering,"[""Circuit Design"", ""MATLAB"", ""Embedded Systems...","[""NPTEL: Electrical Machines"", ""Electrical Pow...","[""Smart Switch Control"", ""DC Motor Speed Contr...",Learn circuit theory and power systems → Work ...
4,5,Civil Engineer,Civil Engineering,"[""AutoCAD"", ""STAAD Pro"", ""Revit"", ""Constructio...","[""NPTEL: Structural Analysis"", ""AutoCAD Civil ...","[""Bridge Design"", ""Apartment Structure Model"",...",Master AutoCAD and STAAD → Learn design princi...


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np, json

model = SentenceTransformer("all-MiniLM-L6-v2")

careers = pd.read_sql("SELECT * FROM careers", conn)

def combine_text(row):
    parts = []
    for key in ["career_name", "category", "required_skills", "recommended_courses", "projects", "roadmap"]:
        val = row[key]
        if val:
            if isinstance(val, str):
                parts.append(val)
            elif isinstance(val, list):
                parts.extend(val)
    return " ".join(parts)

careers["combined_text"] = careers.apply(combine_text, axis=1)
print("✅ Combined text ready:", len(careers))

✅ Combined text ready: 60


In [ ]:
embeddings = model.encode(careers["combined_text"].tolist(), show_progress_bar=True)
np.save("/content/career_embeddings.npy", embeddings)
print("✅ Career embeddings saved.")

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Career embeddings saved.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def recommend_careers(user_id, top_k=5):
    try:
        # Use the already robust get_user_profile_text function
        prof = get_user_profile_text(user_id)
    except ValueError as e:
        print(e)
        return

    name, dept, interests = prof["name"], prof["department"], prof["interests"]

    # Build input text using the combined text from get_user_profile_text
    user_text = prof["user_text"]
    user_emb = model.encode([user_text])

    # Load saved embeddings (assuming they are loaded globally)
    career_embeddings = np.load("/content/career_embeddings.npy")

    # Compute similarity
    sims = cosine_similarity(user_emb, career_embeddings)[0]
    top_idx = sims.argsort()[-top_k:][::-1]
    top_careers = careers.iloc[top_idx]

    # Display results
    print(f"🎯 Top Career Recommendations for {name} ({dept}):\n")
    for i, (idx, row) in enumerate(top_careers.iterrows(), 1):
        print(f"{i}. {row['career_name']} (Similarity: {sims[idx]:.3f})")
        # Use the _list versions for display, which are processed by bkIXcd_e6VZ9
        print(f"   Skills: {row['required_skills_list']}")
        print(f"   Roadmap: {row['roadmap']}")
        print(f"   Courses: {row['recommended_courses_list']}\n")

In [ ]:
# Replace with your actual user_id (check from your users table)
recommend_careers(user_id=1, top_k=5)

User with ID 1 not found. Please provide a valid user ID.


In [ ]:
recommend_careers(user_id)

🎯 Top Career Recommendations for Sasa (CSE):

1. Data Scientist (Similarity: 0.482)
   Skills: ["Python", "Machine Learning", "Statistics", "SQL", "Pandas", "TensorFlow", "Data Visualization", "Communication", "Big Data", "Storytelling"]
   Roadmap: Start with Python and statistics → Learn data preprocessing and visualization → Master machine learning → Work on real-world projects → Learn MLOps and deployment.
   Courses: ["Machine Learning by Andrew Ng (Coursera)", "IBM Data Science Professional Certificate (Coursera)", "NPTEL: Applied Data Science with Python"]

2. Data Analyst (Similarity: 0.468)
   Skills: ["Excel", "SQL", "Python", "Power BI", "Data Visualization", "Statistics"]
   Roadmap: Learn Excel and SQL → Practice Python for analysis → Create dashboards → Perform exploratory data analysis → Build data-driven insights and reports.
   Courses: ["Google Data Analytics Professional Certificate", "IBM Data Analyst Specialization", "NPTEL: Data Analytics with Python"]

3. Softwar

In [ ]:
import pandas as pd

data = [
    {"user_id": 1, "name": "Sashangan", "department": "CSE", "skills": "Python, SQL, ML",
     "projects": "Stock Prediction", "certifications": "NPTEL ML", "interests": "AI, Data Science"},

    {"user_id": 2, "name": "Priya", "department": "ECE", "skills": "C, MATLAB, VLSI",
     "projects": "FPGA Design", "certifications": "NPTEL VLSI", "interests": "Embedded Systems"},

    {"user_id": 3, "name": "Karthik", "department": "MECH", "skills": "AutoCAD, SolidWorks",
     "projects": "Engine Design", "certifications": "Coursera CAD", "interests": "Design, Robotics"},

    {"user_id": 4, "name": "Ananya", "department": "IT", "skills": "JavaScript, React",
     "projects": "Web Portfolio", "certifications": "Coursera React", "interests": "Web Development"},

    {"user_id": 5, "name": "Arjun", "department": "CIVIL", "skills": "AutoCAD, Revit",
     "projects": "Bridge Design", "certifications": "NPTEL Civil Structures", "interests": "Structural Design"},

    {"user_id": 6, "name": "Sneha", "department": "EEE", "skills": "Python, MATLAB",
     "projects": "Smart Grid", "certifications": "Coursera IoT", "interests": "Power Systems"},

    {"user_id": 7, "name": "Riya", "department": "CSE", "skills": "Python, TensorFlow",
     "projects": "Image Classification", "certifications": "Coursera DL", "interests": "AI, Vision Systems"},

    {"user_id": 8, "name": "Manoj", "department": "MECH", "skills": "CNC, CAD",
     "projects": "Gear Automation", "certifications": "NPTEL Mechatronics", "interests": "Automation"},

    {"user_id": 9, "name": "Sanjay", "department": "IT", "skills": "Java, Spring",
     "projects": "Library System", "certifications": "Coursera Java", "interests": "Software Engineering"},

    {"user_id": 10, "name": "Meena", "department": "ECE", "skills": "C++, Embedded C",
     "projects": "IoT Weather Station", "certifications": "NPTEL IoT", "interests": "IoT, Electronics"}
]

df_users = pd.DataFrame(data)
df_users

,user_id,name,department,skills,projects,certifications,interests
0,1,Sashangan,CSE,"Python, SQL, ML",Stock Prediction,NPTEL ML,"AI, Data Science"
1,2,Priya,ECE,"C, MATLAB, VLSI",FPGA Design,NPTEL VLSI,Embedded Systems
2,3,Karthik,MECH,"AutoCAD, SolidWorks",Engine Design,Coursera CAD,"Design, Robotics"
3,4,Ananya,IT,"JavaScript, React",Web Portfolio,Coursera React,Web Development
4,5,Arjun,CIVIL,"AutoCAD, Revit",Bridge Design,NPTEL Civil Structures,Structural Design
5,6,Sneha,EEE,"Python, MATLAB",Smart Grid,Coursera IoT,Power Systems
6,7,Riya,CSE,"Python, TensorFlow",Image Classification,Coursera DL,"AI, Vision Systems"
7,8,Manoj,MECH,"CNC, CAD",Gear Automation,NPTEL Mechatronics,Automation
8,9,Sanjay,IT,"Java, Spring",Library System,Coursera Java,Software Engineering
9,10,Meena,ECE,"C++, Embedded C",IoT Weather Station,NPTEL IoT,"IoT, Electronics"


In [ ]:
df_users.to_csv("/content/dummy_users.csv", index=False)

In [ ]:
import sqlite3
conn = sqlite3.connect("/content/careerbot.db")
cursor = conn.cursor()

In [ ]:
cursor.executescript("""
CREATE TABLE IF NOT EXISTS users (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  username TEXT UNIQUE,
  name TEXT,
  department TEXT,
  interests TEXT
);

CREATE TABLE IF NOT EXISTS skills (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  user_id INTEGER,
  skill TEXT,
  FOREIGN KEY(user_id) REFERENCES users(id)
);

CREATE TABLE IF NOT EXISTS projects (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  user_id INTEGER,
  project TEXT,
  FOREIGN KEY(user_id) REFERENCES users(id)
);

CREATE TABLE IF NOT EXISTS certifications (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  user_id INTEGER,
  certification TEXT,
  FOREIGN KEY(user_id) REFERENCES users(id)
);
""")

conn.commit()
print("✅ All tables created successfully!")

✅ All tables created successfully!


In [ ]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchall()

[('users',),
 ('sqlite_sequence',),
 ('skills',),
 ('projects',),
 ('certifications',),
 ('careers',)]

In [ ]:
for _, row in df_users.iterrows():
    cursor.execute("""
        INSERT INTO users (username, name, department, interests)
        VALUES (?, ?, ?, ?)
    """, (
        row["name"].lower(),
        row["name"],
        row["department"],
        row["interests"]
    ))

    user_id = cursor.lastrowid

    # Insert skills
    for skill in row["skills"].split(","):
        cursor.execute("INSERT INTO skills (user_id, skill) VALUES (?, ?)", (user_id, skill.strip()))

    # Insert projects
    for proj in row["projects"].split(","):
        cursor.execute("INSERT INTO projects (user_id, project) VALUES (?, ?)", (user_id, proj.strip()))

    # Insert certifications
    for cert in row["certifications"].split(","):
        cursor.execute("INSERT INTO certifications (user_id, certification) VALUES (?, ?)", (user_id, cert.strip()))

conn.commit()
print("✅ Dummy user data successfully inserted!")

✅ Dummy user data successfully inserted!


In [ ]:
print("Users:")
print(pd.read_sql("SELECT * FROM users", conn))

print("\nSkills:")
print(pd.read_sql("SELECT * FROM skills", conn))

Users:
   id   username       name department             interests
0   1  sashangan  Sashangan        CSE      AI, Data Science
1   2      priya      Priya        ECE      Embedded Systems
2   3    karthik    Karthik       MECH      Design, Robotics
3   4     ananya     Ananya         IT       Web Development
4   5      arjun      Arjun      CIVIL     Structural Design
5   6      sneha      Sneha        EEE         Power Systems
6   7       riya       Riya        CSE    AI, Vision Systems
7   8      manoj      Manoj       MECH            Automation
8   9     sanjay     Sanjay         IT  Software Engineering
9  10      meena      Meena        ECE      IoT, Electronics

Skills:
    id  user_id       skill
0    1        1      Python
1    2        1         SQL
2    3        1          ML
3    4        2           C
4    5        2      MATLAB
5    6        2        VLSI
6    7        3     AutoCAD
7    8        3  SolidWorks
8    9        4  JavaScript
9   10        4       React
10  1

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import json
import random
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD

# Connect to your DB (adjust path if needed)
DB_PATH = "/content/careerbot.db"
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

In [ ]:
def safe_load_json_field(x):
    """Try to load JSON list; if not JSON, return cleaned string/list."""
    if x is None:
        return []
    # If already a list
    if isinstance(x, list):
        return x
    # If it's a numeric / other, convert to string
    if not isinstance(x, str):
        return [str(x)]
    s = x.strip()
    # Try JSON
    try:
        parsed = json.loads(s)
        # ensure list of strings
        if isinstance(parsed, list):
            return [str(i) for i in parsed]
        else:
            return [str(parsed)]
    except:
        # fallback: split on commas
        if "," in s:
            return [p.strip() for p in s.split(",") if p.strip()]
        # fallback: return the whole string
        return [s] if s else []

# Read careers table
careers = pd.read_sql("SELECT * FROM careers", conn)

# Normalize columns we will use
for col in ["required_skills", "recommended_courses", "projects", "category", "roadmap", "career_name"]:
    if col not in careers.columns:
        careers[col] = ""

# Ensure structured lists
careers["required_skills_list"] = careers["required_skills"].apply(safe_load_json_field)
careers["recommended_courses_list"] = careers["recommended_courses"].apply(safe_load_json_field)
careers["projects_list"] = careers["projects"].apply(safe_load_json_field)

# Combine text for each career
def combine_text(row):
    pieces = []
    for c in ["career_name", "category", "roadmap"]:
        if row.get(c):
            pieces.append(str(row[c]))
    pieces.extend([str(x) for x in row["required_skills_list"]])
    pieces.extend([str(x) for x in row["recommended_courses_list"]])
    pieces.extend([str(x) for x in row["projects_list"]])
    return " . ".join([p for p in pieces if p])
careers["combined_text"] = careers.apply(combine_text, axis=1)

print("Loaded careers:", len(careers))
careers.head(2)

Loaded careers: 60


,id,career_name,category,required_skills,recommended_courses,projects,roadmap,required_skills_list,recommended_courses_list,projects_list,combined_text
0,1,Data Scientist,Artificial Intelligence,"[""Python"", ""Machine Learning"", ""Statistics"", ""...","[""Machine Learning by Andrew Ng (Coursera)"", ""...","[""Customer Churn Prediction"", ""Sales Forecasti...",Start with Python and statistics → Learn data ...,"[Python, Machine Learning, Statistics, SQL, Pa...","[Machine Learning by Andrew Ng (Coursera), IBM...","[Customer Churn Prediction, Sales Forecasting,...",Data Scientist . Artificial Intelligence . Sta...
1,2,Software Engineer,Computer Science,"[""Python"", ""Java"", ""C++"", ""Data Structures"", ""...","[""CS50: Introduction to Computer Science (edX)...","[""Portfolio Website"", ""Chat Application"", ""Lib...",Learn programming languages and DSA → Build sm...,"[Python, Java, C++, Data Structures, Algorithm...","[CS50: Introduction to Computer Science (edX),...","[Portfolio Website, Chat Application, Library ...",Software Engineer . Computer Science . Learn p...


In [ ]:
def get_user_profile_text(user_id):
    # fetch basic info
    r = conn.execute("SELECT name, department, interests FROM users WHERE id=?", (user_id,)).fetchone()
    if not r:
        raise ValueError(f"user_id {user_id} not found")
    name, department, interests = r
    # fetch skills, projects, certifications
    skills = [x[0] for x in conn.execute("SELECT skill FROM skills WHERE user_id=?", (user_id,)).fetchall()]
    projects = [x[0] for x in conn.execute("SELECT project FROM projects WHERE user_id=?", (user_id,)).fetchall()]
    certs = [x[0] for x in conn.execute("SELECT certification FROM certifications WHERE user_id=?", (user_id,)).fetchall()]

    # build text
    parts = [str(department) or "", interests or "", " ".join(skills), " ".join(projects), " ".join(certs)]
    user_text = " . ".join([p for p in parts if p])
    return {
        "name": name,
        "department": department,
        "interests": interests,
        "skills": skills,
        "projects": projects,
        "certifications": certs,
        "user_text": user_text
    }

# quick test (replace 1 with an actual user id you have)
# print(get_user_profile_text(1))

ALGORITHM 1 — TF-IDF + Cosine Similarity

In [ ]:
tfidf = TfidfVectorizer(stop_words="english", max_features=20000)
tfidf_matrix = tfidf.fit_transform(careers["combined_text"].tolist())
print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (60, 735)


In [ ]:
def recommend_tfidf_for_user(user_id, top_k=5):
    prof = get_user_profile_text(user_id)
    user_vec = tfidf.transform([prof["user_text"]])
    sims = cosine_similarity(user_vec, tfidf_matrix)[0]
    top_idx = sims.argsort()[-top_k:][::-1]
    results = []
    for i in top_idx:
        row = careers.iloc[i]
        results.append({
            "career_name": row["career_name"],
            "method": "tfidf",
            "score": float(sims[i]),
            "required_skills": row["required_skills_list"],
            "recommended_courses": row["recommended_courses_list"],
            "roadmap": row["roadmap"]
        })
    return results

ALGORITHM 2 — SentenceTransformer Embeddings

In [ ]:
EMB_PATH = "/content/career_embeddings.npy"
model = SentenceTransformer("all-MiniLM-L6-v2")

if Path(EMB_PATH).exists():
    career_emb = np.load(EMB_PATH)
    print("Loaded existing embeddings:", career_emb.shape)
else:
    career_emb = model.encode(careers["combined_text"].tolist(), show_progress_bar=True, normalize_embeddings=True)
    np.save(EMB_PATH, career_emb)
    print("Computed and saved embeddings:", career_emb.shape)

Loaded existing embeddings: (60, 384)


In [ ]:
def recommend_embedding_for_user(user_id, top_k=5):
    prof = get_user_profile_text(user_id)
    user_emb = model.encode([prof["user_text"]], normalize_embeddings=True)
    sims = cosine_similarity(user_emb, career_emb)[0]
    top_idx = sims.argsort()[-top_k:][::-1]
    results = []
    for i in top_idx:
        row = careers.iloc[i]
        results.append({
            "career_name": row["career_name"],
            "method": "embedding",
            "score": float(sims[i]),
            "required_skills": row["required_skills_list"],
            "recommended_courses": row["recommended_courses_list"],
            "roadmap": row["roadmap"]
        })
    return results

ALGORITHM 3 — Rule-Based Scoring

In [ ]:
def compute_rule_score_for_row(user_profile, career_row):
    score = 0
    # branch/category match (strong weight)
    career_category = str(career_row.get("category","") or "").lower()
    if user_profile["department"] and user_profile["department"].lower() in career_category:
        score += 5
    # exact skill matches (each skill gives 2 points)
    required = [s.lower() for s in career_row["required_skills_list"]]
    for us in user_profile["skills"]:
        if us.lower() in required:
            score += 2
    # certifications boost
    for cert in user_profile["certifications"]:
        if cert.lower() in str(career_row.get("recommended_courses","")).lower():
            score += 1
    # small boost if interest word appears in career text
    if user_profile["interests"]:
        for word in str(user_profile["interests"]).split(","):
            w = word.strip().lower()
            if w and w in career_row["combined_text"].lower():
                score += 1
    return score

def recommend_rule_based_for_user(user_id, top_k=5):
    prof = get_user_profile_text(user_id)
    scores = []
    for _, row in careers.iterrows():
        s = compute_rule_score_for_row(prof, row)
        scores.append(s)
    careers["rule_score_temp"] = scores
    top = careers.sort_values(by="rule_score_temp", ascending=False).head(top_k)
    results = []
    for _, row in top.iterrows():
        results.append({
            "career_name": row["career_name"],
            "method": "rule_based",
            "score": int(row["rule_score_temp"]),
            "required_skills": row["required_skills_list"],
            "recommended_courses": row["recommended_courses_list"],
            "roadmap": row["roadmap"]
        })
    return results

ALGORITHM 4 — KMeans Clusterin

In [ ]:
N_CLUSTERS = 6
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
kmeans.fit(tfidf_matrix)
careers["cluster"] = kmeans.labels_
print("Assigned clusters, sample counts per cluster:")
print(careers["cluster"].value_counts())

Assigned clusters, sample counts per cluster:
cluster
1    17
2    11
0    11
5    10
3     6
4     5
Name: count, dtype: int64


In [ ]:
def recommend_kmeans_for_user(user_id, top_k=5):
    prof = get_user_profile_text(user_id)
    user_vec = tfidf.transform([prof["user_text"]])
    cluster = int(kmeans.predict(user_vec)[0])
    candidates = careers[careers["cluster"] == cluster].copy()
    # rank by embedding similarity if available, else TF-IDF similarity
    if 'career_emb' in globals():
        # compute similarity against career_emb rows of this cluster
        idxs = candidates.index.tolist()
        cluster_emb = career_emb[idxs]
        user_emb = model.encode([prof["user_text"]], normalize_embeddings=True)
        sims = cosine_similarity(user_emb, cluster_emb)[0]
        order = np.argsort(sims)[-top_k:][::-1]
        selected_idx = [idxs[i] for i in order]
    else:
        user_vec = tfidf.transform([prof["user_text"]])
        sims = cosine_similarity(user_vec, tfidf.transform(candidates["combined_text"]))[0]
        order = np.argsort(sims)[-top_k:][::-1]
        selected_idx = candidates.index[order].tolist()

    results = []
    for i in selected_idx:
        row = careers.loc[i]
        results.append({
            "career_name": row["career_name"],
            "method": "kmeans",
            "score": None,
            "cluster": int(row["cluster"]),
            "required_skills": row["required_skills_list"],
            "recommended_courses": row["recommended_courses_list"],
            "roadmap": row["roadmap"]
        })
    return results

ALGORITHM 5 — SVD (TruncatedSVD on TF-IDF)

In [ ]:
SVD_COMPONENTS = 50
svd = TruncatedSVD(n_components=min(SVD_COMPONENTS, tfidf_matrix.shape[1]-1), random_state=42)
career_svd = svd.fit_transform(tfidf_matrix)
print("Career SVD shape:", career_svd.shape)

Career SVD shape: (60, 50)


In [ ]:
def recommend_svd_for_user(user_id, top_k=5):
    prof = get_user_profile_text(user_id)
    user_vec = svd.transform(tfidf.transform([prof["user_text"]]))
    sims = cosine_similarity(user_vec, career_svd)[0]
    top_idx = sims.argsort()[-top_k:][::-1]
    results = []
    for i in top_idx:
        row = careers.iloc[i]
        results.append({
            "career_name": row["career_name"],
            "method": "svd",
            "score": float(sims[i]),
            "required_skills": row["required_skills_list"],
            "recommended_courses": row["recommended_courses_list"],
            "roadmap": row["roadmap"]
        })
    return results

In [ ]:
def recommend_all_for_user(user_id, top_k=5):
    tfidf_res = recommend_tfidf_for_user(user_id, top_k=top_k)
    emb_res = recommend_embedding_for_user(user_id, top_k=top_k)
    rule_res = recommend_rule_based_for_user(user_id, top_k=top_k)
    kmeans_res = recommend_kmeans_for_user(user_id, top_k=top_k)
    svd_res = recommend_svd_for_user(user_id, top_k=top_k)

    return {
        "tfidf": tfidf_res,
        "embedding": emb_res,
        "rule_based": rule_res,
        "kmeans": kmeans_res,
        "svd": svd_res
    }

# Example usage (replace user_id with an existing user)
# results = recommend_all_for_user(user_id=1, top_k=5)
# for algo, res in results.items():
#     print("=== ", algo.upper(), "===")
#     for r in res:
#         print(r)
#     print()

In [ ]:
user_id = 1  # change to a real user id from your users table
all_res = recommend_all_for_user(user_id=user_id, top_k=5)

for algo, res in all_res.items():
    print("\n" + "="*8 + f" {algo.upper()} " + "="*8)
    for idx, r in enumerate(res, start=1):
        print(f"{idx}. {r['career_name']}  (score: {r.get('score')})")
        print(f"   required_skills: {r['required_skills']}")
        print(f"   recommended_courses: {r['recommended_courses'][:3] if r.get('recommended_courses') else []}")
        print(f"   roadmap: {r.get('roadmap')}\n")


======== TFIDF ========
1. Data Analyst  (score: 0.29046897638449864)
   required_skills: ['Excel', 'SQL', 'Python', 'Power BI', 'Data Visualization', 'Statistics']
   recommended_courses: ['Google Data Analytics Professional Certificate', 'IBM Data Analyst Specialization', 'NPTEL: Data Analytics with Python']
   roadmap: Learn Excel and SQL → Practice Python for analysis → Create dashboards → Perform exploratory data analysis → Build data-driven insights and reports.

2. Data Scientist  (score: 0.29025636859720766)
   required_skills: ['Python', 'Machine Learning', 'Statistics', 'SQL', 'Pandas', 'TensorFlow', 'Data Visualization', 'Communication', 'Big Data', 'Storytelling']
   recommended_courses: ['Machine Learning by Andrew Ng (Coursera)', 'IBM Data Science Professional Certificate (Coursera)', 'NPTEL: Applied Data Science with Python']
   roadmap: Start with Python and statistics → Learn data preprocessing and visualization → Master machine learning → Work on real-world projects 

In [ ]:
!pip install rank-bm25

ALGORITHM 6 — BM25 (Best Matching 25)

In [ ]:
from rank_bm25 import BM25Okapi

corpus = [text.split() for text in careers["combined_text"].tolist()]
bm25 = BM25Okapi(corpus)

In [ ]:
def recommend_bm25_for_user(user_id, top_k=5):
    prof = get_user_profile_text(user_id)
    query = prof["user_text"].split()

    scores = bm25.get_scores(query)  # BM25 scores
    top_idx = np.argsort(scores)[-top_k:][::-1]

    results = []
    for i in top_idx:
        row = careers.iloc[i]
        results.append({
            "career_name": row["career_name"],
            "method": "BM25",
            "score": float(scores[i]),
            "required_skills": row["required_skills_list"],
            "recommended_courses": row["recommended_courses_list"],
            "roadmap": row["roadmap"],
        })
    return results

In [ ]:
tfidf_ngram = TfidfVectorizer(stop_words="english", ngram_range=(1,3))
tfidf_ngram_matrix = tfidf_ngram.fit_transform(careers["combined_text"])

ALGORITHM 7 — BIGRAM TF-IDF / TRIGRAM TF-IDF

In [ ]:
tfidf_ngram = TfidfVectorizer(stop_words="english", ngram_range=(1,3))
tfidf_ngram_matrix = tfidf_ngram.fit_transform(careers["combined_text"])

In [ ]:
def recommend_tfidf_ngram_for_user(user_id, top_k=5):
    prof = get_user_profile_text(user_id)
    user_vec = tfidf_ngram.transform([prof["user_text"]])
    sims = cosine_similarity(user_vec, tfidf_ngram_matrix)[0]

    top_idx = sims.argsort()[-top_k:][::-1]
    results = []
    for i in top_idx:
        row = careers.iloc[i]
        results.append({
            "career_name": row["career_name"],
            "method": "TFIDF_NGRAM",
            "score": float(sims[i]),
            "required_skills": row["required_skills_list"],
            "recommended_courses": row["recommended_courses_list"],
            "roadmap": row["roadmap"]
        })
    return results

ALGORITHM 8 — FastText Embeddings

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 92.6 MB/s eta 0:00:00


In [ ]:
from gensim.models import FastText

sentences = [text.split() for text in careers["combined_text"]]
fasttext_model = FastText(sentences, vector_size=100, window=5, min_count=1, workers=4)

In [ ]:
def vectorize_fasttext(text):
    words = text.split()
    vectors = [fasttext_model.wv[w] for w in words if w in fasttext_model.wv]
    if len(vectors) == 0:
        return np.zeros(100)
    return np.mean(vectors, axis=0)

career_ft_vectors = np.array([vectorize_fasttext(t) for t in careers["combined_text"]])

def recommend_fasttext_for_user(user_id, top_k=5):
    prof = get_user_profile_text(user_id)
    user_vec = vectorize_fasttext(prof["user_text"]).reshape(1, -1)
    sims = cosine_similarity(user_vec, career_ft_vectors)[0]

    top_idx = sims.argsort()[-top_k:][::-1]
    return careers.iloc[top_idx][["career_name", "required_skills_list", "roadmap"]]

ALGORITHM 9 — Doc2Vec Embeddings

In [ ]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

documents = [TaggedDocument(words=text.split(), tags=[i]) for i, text in enumerate(careers["combined_text"])]
doc2vec = Doc2Vec(documents, vector_size=100, window=5, min_count=1, workers=4)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_doc2vec_for_user(user_id, top_k=5):
    prof = get_user_profile_text(user_id)
    user_vec = doc2vec.infer_vector(prof["user_text"].split())

    # Corrected line to use sklearn's cosine_similarity with the actual document vectors
    sims = cosine_similarity([user_vec], doc2vec.dv.vectors)[0]
    top_idx = np.argsort(sims)[-top_k:][::-1]

    # Prepare results in a consistent dictionary format
    results = []
    for i in top_idx:
        row = careers.iloc[i]
        results.append({
            "career_name": row["career_name"],
            "method": "doc2vec",
            "score": float(sims[i]),
            "required_skills": row["required_skills_list"],
            "recommended_courses": row["recommended_courses_list"],
            "roadmap": row["roadmap"]
        })
    return results

Keyword Overlap Index

In [ ]:
def keyword_overlap(a, b):
    set_a = set(a.lower().split())
    set_b = set(b.lower().split())
    return len(set_a & set_b)

In [ ]:
def recommend_keyword_overlap_for_user(user_id, top_k=5):
    prof = get_user_profile_text(user_id)
    user_text = prof["user_text"]

    scores = []
    for idx, row in careers.iterrows():
        score = keyword_overlap(user_text, row["combined_text"])
        scores.append((idx, score))

    scores = sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]
    top_idx = [x[0] for x in scores]

    return careers.iloc[top_idx][["career_name", "required_skills_list", "roadmap"]]

In [ ]:
def print_algo_results(title, results):
    print("\n" + "="*10 + f" {title} " + "="*10)
    for i, row in enumerate(results, start=1):
        if isinstance(row, dict):
            name = row.get("career_name")
            score = row.get("score")
            # Check for both 'required_skills' and 'required_skills_list'
            req = row.get("required_skills") or row.get("required_skills_list", [])
            roadmap = row.get("roadmap")
            # Check for both 'recommended_courses' and 'recommended_courses_list'
            courses = row.get("recommended_courses") or row.get("recommended_courses_list", [])
        else:
            # This path is typically for DataFrame rows, extract directly
            name = row["career_name"]
            score = None
            req = row.get("required_skills_list", [])
            roadmap = row.get("roadmap")
            courses = row.get("recommended_courses_list", [])

        print(f"{i}. {name}  (score: {score})")
        print(f"   required_skills: {req[:5]}")
        print(f"   recommended_courses: {courses[:3] if courses else []}")
        print(f"   roadmap: {roadmap}\n")


# -----------------------------
# RUN ALL LAST 5 ALGORITHMS
# -----------------------------
user_id = 1   # change to your test user

print("\U0001F50D Running LAST 5 ALGORITHMS (6-10) for user:", user_id)

# 6. BM25
bm25_res = recommend_bm25_for_user(user_id, top_k=5)
print_algo_results("BM25", bm25_res)

# 7. N-GRAM TF-IDF
ngram_res = recommend_tfidf_ngram_for_user(user_id, top_k=5)
print_algo_results("TF-IDF NGRAM (1–3)", ngram_res)

# 8. FASTTEXT
fast_res = recommend_fasttext_for_user(user_id, top_k=5)
print_algo_results("FASTTEXT", fast_res.to_dict(orient="records"))

# 9. DOC2VEC
doc2vec_res = recommend_doc2vec_for_user(user_id, top_k=5)
print_algo_results("DOC2VEC", doc2vec_res)

# 10. KEYWORD OVERLAP
keyword_res = recommend_keyword_overlap_for_user(user_id, top_k=5)
print_algo_results("KEYWORD OVERLAP", keyword_res.to_dict(orient="records"))

🔍 Running LAST 5 ALGORITHMS (6-10) for user: 1

========== BM25 ==========
1. Data Scientist  (score: 18.709192992351884)
   required_skills: ['Python', 'Machine Learning', 'Statistics', 'SQL', 'Pandas']
   recommended_courses: ['Machine Learning by Andrew Ng (Coursera)', 'IBM Data Science Professional Certificate (Coursera)', 'NPTEL: Applied Data Science with Python']
   roadmap: Start with Python and statistics → Learn data preprocessing and visualization → Master machine learning → Work on real-world projects → Learn MLOps and deployment.

2. Data Analyst  (score: 17.032869549479187)
   required_skills: ['Excel', 'SQL', 'Python', 'Power BI', 'Data Visualization']
   recommended_courses: ['Google Data Analytics Professional Certificate', 'IBM Data Analyst Specialization', 'NPTEL: Data Analytics with Python']
   roadmap: Learn Excel and SQL → Practice Python for analysis → Create dashboards → Perform exploratory data analysis → Build data-driven insights and reports.

3. Agricultural 

In [ ]:
def auto_generate_ground_truth():
    ground_truth = {}

    for user_id in range(1, 11):  # Adjust for number of users
        prof = get_user_profile_text(user_id)
        branch = prof["department"].lower()
        skills = [s.lower() for s in prof["skills"]]
        interests = str(prof["interests"]).lower()

        true_roles = []

        # -------------------------
        # CSE USERS
        # -------------------------
        if branch == "cse":
            if any(s in skills for s in ["python", "ml", "ai", "sql", "data"]):
                true_roles += ["Data Scientist", "ML Engineer", "AI Engineer"]
            if "web" in interests:
                true_roles += ["Frontend Developer", "Backend Developer", "Full Stack Developer"]

        # -------------------------
        # ECE USERS
        # -------------------------
        if branch == "ece":
            if any(s in skills for s in ["vlsi", "embedded", "matlab", "verilog"]):
                true_roles += ["Embedded Engineer", "VLSI Engineer", "IoT Engineer"]

        # -------------------------
        # MECH USERS
        # -------------------------
        if branch == "mech":
            if any(s in skills for s in ["cad", "solidworks", "ansys"]):
                true_roles += ["Mechanical Design Engineer"]

        # -------------------------
        # CIVIL USERS
        # -------------------------
        if branch == "civil":
            if any(s in skills for s in ["autocad", "revit"]):
                true_roles += ["Structural Engineer"]

        # -------------------------
        # EEE USERS
        # -------------------------
        if branch == "eee":
            if any(s in skills for s in ["power", "matlab", "simulink"]):
                true_roles += ["Power Systems Engineer"]

        ground_truth[user_id] = list(set(true_roles))  # remove duplicates

    return ground_truth


ground_truth = auto_generate_ground_truth()
ground_truth

{1: ['ML Engineer', 'Data Scientist', 'AI Engineer'],
 2: ['IoT Engineer', 'Embedded Engineer', 'VLSI Engineer'],
 3: ['Mechanical Design Engineer'],
 4: [],
 5: ['Structural Engineer'],
 6: ['Power Systems Engineer'],
 7: ['ML Engineer', 'Data Scientist', 'AI Engineer'],
 8: ['Mechanical Design Engineer'],
 9: [],
 10: []}

In [ ]:
import numpy as np

def precision_at_k(recommended, relevant, k=5):
    rec_k = recommended[:k]
    return len(set(rec_k) & set(relevant)) / k

def recall_at_k(recommended, relevant, k=5):
    rec_k = recommended[:k]
    return len(set(rec_k) & set(relevant)) / len(relevant) if len(relevant) > 0 else 0

def average_precision(recommended, relevant, k=5):
    score = 0
    hits = 0
    relevant_set = set(relevant)

    for i, r in enumerate(recommended[:k], start=1):
        if r in relevant_set:
            hits += 1
            score += hits / i

    return score / len(relevant) if len(relevant) > 0 else 0

def ndcg_at_k(recommended, relevant, k=5):
    dcg = 0
    idcg = 0

    for i, r in enumerate(recommended[:k], start=1):
        if r in relevant:
            dcg += 1 / np.log2(i + 1)

    for i in range(1, min(k, len(relevant)) + 1):
        idcg += 1 / np.log2(i + 1)

    return dcg / idcg if idcg > 0 else 0

In [ ]:
def evaluate_algorithm(algorithm_func, ground_truth, top_k=5):
    p_scores = []
    r_scores = []
    map_scores = []
    ndcg_scores = []

    for user_id, relevant in ground_truth.items():
        if len(relevant) == 0:
            continue

        results = algorithm_func(user_id, top_k)
        recommended = [r["career_name"] for r in results]

        p_scores.append(precision_at_k(recommended, relevant, top_k))
        r_scores.append(recall_at_k(recommended, relevant, top_k))
        map_scores.append(average_precision(recommended, relevant, top_k))
        ndcg_scores.append(ndcg_at_k(recommended, relevant, top_k))

    return {
        "Precision@5": np.mean(p_scores),
        "Recall@5": np.mean(r_scores),
        "MAP@5": np.mean(map_scores),
        "nDCG@5": np.mean(ndcg_scores)
    }

In [ ]:
def recommend_fasttext_for_user(user_id, top_k=5):
    prof = get_user_profile_text(user_id)
    user_vec = vectorize_fasttext(prof["user_text"]).reshape(1, -1)
    sims = cosine_similarity(user_vec, career_ft_vectors)[0]

    top_idx = sims.argsort()[-top_k:][::-1]

    results = []
    for i in top_idx:
        row = careers.iloc[i]
        results.append({
            "career_name": row["career_name"],
            "score": float(sims[i]),
            "required_skills": row.get("required_skills_list", []),
            "recommended_courses": row.get("recommended_courses_list", []),
            "roadmap": row.get("roadmap", "")
        })

    return results

In [ ]:
def recommend_doc2vec_for_user(user_id, top_k=5):
    prof = get_user_profile_text(user_id)
    user_vec = doc2vec.infer_vector(prof["user_text"].split())

    sims = doc2vec.dv.cosine_similarities(user_vec, doc2vec.dv)
    top_idx = np.argsort(sims)[-top_k:][::-1]

    results = []
    for i in top_idx:
        row = careers.iloc[i]
        results.append({
            "career_name": row["career_name"],
            "score": float(sims[i]),
            "required_skills": row.get("required_skills_list", []),
            "recommended_courses": row.get("recommended_courses_list", []),
            "roadmap": row.get("roadmap", "")
        })

    return results

In [ ]:
def recommend_keyword_overlap_for_user(user_id, top_k=5):
    prof = get_user_profile_text(user_id)
    user_text = prof["user_text"]

    scores = []
    for idx, row in careers.iterrows():
        score = keyword_overlap(user_text, row["combined_text"])
        scores.append((idx, score))

    scores = sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]
    top_idx = [x[0] for x in scores]

    results = []
    for i in top_idx:
        row = careers.iloc[i]
        results.append({
            "career_name": row["career_name"],
            "score": float(scores[top_idx.index(i)][1]),
            "required_skills": row.get("required_skills_list", []),
            "roadmap": row.get("roadmap", "")
        })

    return results

In [ ]:
results["FastText"] = evaluate_algorithm(recommend_fasttext_for_user, ground_truth)


In [ ]:
results = {}
results_df = pd.DataFrame(results).T

In [ ]:
# -----------------------------
# SAFELY BUILD RESULTS TABLE
# -----------------------------
results_df = pd.DataFrame(results).T  # Convert dict → table

# Rename metrics for clarity
results_df = results_df.rename(columns={
    "Precision@5": "Precision@5",
    "Recall@5": "Recall@5",
    "MAP@5": "MAP@5",
    "nDCG@5": "nDCG@5"
})

# Round values for better readability
results_df = results_df.round(3)

# Display results
print("📊 FINAL COMPARISON TABLE (10 Algorithms)")
display(results_df)


# -----------------------------
# FIND THE BEST ALGORITHM
# -----------------------------
if "MAP@5" in results_df.columns:
    best_algo = results_df["MAP@5"].idxmax()
    print("\n🏆 BEST PERFORMING ALGORITHM:", best_algo)
else:
    print("\n⚠️ ERROR: MAP@5 column missing. Check evaluation output.")

📊 FINAL COMPARISON TABLE (10 Algorithms)


,Precision@5,Recall@5,MAP@5,nDCG@5
TF-IDF,0.114,0.286,0.262,0.300
Embeddings,0.143,0.429,0.305,0.368
Rule-Based,0.143,0.429,0.303,0.378
KMeans,0.086,0.238,0.238,0.257
SVD,0.114,0.286,0.262,0.300
BM25,0.143,0.333,0.290,0.348
N-Gram TF-IDF,0.114,0.286,0.262,0.300
FastText,0.029,0.143,0.143,0.143



🏆 BEST PERFORMING ALGORITHM: Embeddings
